# VERIONCO — Notebook 03: Clinical Extraction (Full Dataset, 478 Cases)

**Purpose:** parse TCGA clinical patient/sample data, build structured
clinical evidence per patient, cross-check the joined cohort from Notebook 00,
and produce a flat clinical table for downstream integration.

**Design principles:**
- Reads `cohort_joined_wsi.csv` from Notebook 00 (full dataset: 478 cases / 471 clinical cases).
- Cross-checks patient overlap with Notebooks 01 and 02.
- Cross-checks modality availability from Notebook 00.
- Every clinical record carries `patient_id` for leakage-safe joining.
- Every field is captured with provenance; missing values stay null, never
  fabricated.
- Per-patient clinical JSON is written to a directory so downstream notebooks
  do not need to parse a monolithic file.
- Adds a `clinical_features.csv` with a documented, minimal feature schema
  for planners and safety-gate notebooks.
- Clinical data limitations are documented in `clinical_data_limitation.json`.

## INPUTS

| Source | Path | Used for |
|--------|------|----------|
| Notebook 00 | `.../VERIONCO_outputs/00/` | `cohort_joined_wsi.csv`, `patient_master.csv`, `modality_completeness.csv`, `case_manifest_validated.csv` |
| Notebook 01 | `.../pathology-pipeline-01/` | `patch_manifest.csv` for patient-overlap check |
| Notebook 02 | `.../genomic-dataset-02/` | `genomic_patient_summary.csv`, `genomic_qc_report.json` |
| TCGA clinical patient | `.../tcgadataset/.../data_clinical_patient.txt` | Demographics, staging, survival |
| TCGA clinical sample  | `.../tcgadataset/.../data_clinical_sample.txt`  | Sample-level attributes, biomarkers |

**Data-format note:** cBioPortal clinical files begin with metadata lines
starting with `#`. The actual header row is the first non-`#` line.

**Data caveat:** cBioPortal LUAD clinical files do NOT contain drug-level
treatment data. Only `HISTORY_NEOADJUVANT_TRTYN` is available. Do not
invent treatment histories.

## CONFIGURATION

Edit only the `PATHS` block to match your Kaggle attachment names.

In [1]:
# ============================================================
# CONFIGURATION  ← edit only this cell
# ============================================================
from pathlib import Path
import os, gc, sys, json, hashlib, warnings, time, traceback
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Detect environment
IS_KAGGLE = Path("/kaggle").exists()

def find_dir(candidates, fallback):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    return Path(fallback)

OUTPUT_ROOT = Path("/kaggle/working/VERIONCO_outputs") if IS_KAGGLE else Path("./VERIONCO_outputs")
NB_OUTPUT       = OUTPUT_ROOT / "03"
NB_OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints" / "03"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

PATHS = {
    "nb00": find_dir([
        "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00",
        "/kaggle/input/verionco-outputs-00/00",
        "/kaggle/input/verionco-outputs-00",
        "/kaggle/input/verionco-outputs-local/00",
        "/kaggle/working/VERIONCO_outputs/00",
        "./VERIONCO_outputs/00"
    ], fallback="./VERIONCO_outputs/00" if not IS_KAGGLE else "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00"),
    
    "nb01": find_dir([
        "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-01",
        "/kaggle/input/verionco-outputs-01",
        "/kaggle/input/verionco-outputs-local/01",
        "/kaggle/working/VERIONCO_outputs/01",
        "./VERIONCO_outputs/01"
    ], fallback="./VERIONCO_outputs/01" if not IS_KAGGLE else "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-01"),
    
    "nb02": find_dir([
        "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-02",
        "/kaggle/input/verionco-outputs-02",
        "/kaggle/input/verionco-outputs-local/02",
        "/kaggle/working/VERIONCO_outputs/02",
        "./VERIONCO_outputs/02"
    ], fallback="./VERIONCO_outputs/02" if not IS_KAGGLE else "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-02"),
    
    "tcga": find_dir([
        "/kaggle/input/datasets/abhigyansinha09/tcgadataset",
        "/kaggle/input/tcgadataset",
        "/kaggle/input/datasets/abhigyansinha09/tcga",
        "/kaggle/input/tcga",
        "./data/tcga"
    ], fallback="./data/tcga" if not IS_KAGGLE else "/kaggle/input/datasets/abhigyansinha09/tcgadataset"),
}

EXPECTED_MIN_CASES = 10
EXPECTED_MAX_CASES = 550
PROJECT  = "VERIONCO"
NOTEBOOK = "03_clinical_extraction"
HIGH_MISSINGNESS_THRESHOLD = 50.0   # percent

for k, v in PATHS.items():
    print(f"{k:8s} | {v} | exists={v.exists()}")

assert PATHS["nb00"].exists(), f"Notebook 00 root missing: {PATHS['nb00']}"
assert PATHS["tcga"].exists(), f"TCGA root missing: {PATHS['tcga']}"
if not PATHS["nb01"].exists():
    warnings.warn(f"Notebook 01 root missing: {PATHS['nb01']} — overlap check will be skipped.")
if not PATHS["nb02"].exists():
    warnings.warn(f"Notebook 02 root missing: {PATHS['nb02']} — genomic cross-check will be skipped.")

print(f"\nOutput:      {NB_OUTPUT}")
print(f"Checkpoints: {CHECKPOINT_ROOT}")


nb00     | VERIONCO_outputs\00 | exists=True
nb01     | VERIONCO_outputs\01 | exists=False
nb02     | VERIONCO_outputs\02 | exists=True
tcga     | data\tcga | exists=True

Output:      VERIONCO_outputs\03
Checkpoints: VERIONCO_outputs\checkpoints\03


<string>:78: UserWarning: Notebook 01 root missing: VERIONCO_outputs\01 — overlap check will be skipped.


## Environment check

In [2]:
import importlib.util
packages = ["pandas", "numpy", "matplotlib", "seaborn"]
rows = [{"package": p, "available": importlib.util.find_spec(p) is not None}
        for p in packages]
display(pd.DataFrame(rows))

print(json.dumps({
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
}, indent=2))

,package,available
0,pandas,True
1,numpy,True
2,matplotlib,True
3,seaborn,True


{
  "timestamp_utc": "2026-10-19T13:20:34.661549+00:00",
  "python": "3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]"
}


## LOAD — cohort from Notebook 00 + cross-checks

In [3]:
# ============================================================
# LOAD — cohort from Notebook 00 + cross-checks with NB01/NB02
# ============================================================
t0 = time.time()

cohort_path = PATHS["nb00"] / "cohort_joined_wsi.csv"
assert cohort_path.exists(), f"Missing {cohort_path}. Run corrected Notebook 00 first."

cohort = pd.read_csv(cohort_path)
print(f"cohort_joined_wsi.csv: {cohort.shape}")

if "filename" in cohort.columns and "file_name" not in cohort.columns:
    cohort = cohort.rename(columns={"filename": "file_name"})

if "patient_id" not in cohort.columns:
    raise ValueError("cohort_joined_wsi.csv missing 'patient_id' column.")

if cohort["patient_id"].isna().any():
    raise ValueError("cohort_joined_wsi.csv has missing patient_id values.")

patient_set = set(cohort["patient_id"].astype(str).unique())
n_cases     = len(cohort)
n_patients  = len(patient_set)
print(f"cohort: {n_cases} cases | {n_patients} unique patients")

# ---- Cross-check 1: NB01 patch manifest --------------------
nb01_patients = set()
if PATHS["nb01"].exists():
    patch_hits = list(PATHS["nb01"].rglob("patch_manifest.csv"))
    if patch_hits:
        try:
            pm = pd.read_csv(patch_hits[0], usecols=["patient_id"])
            nb01_patients = set(pm["patient_id"].astype(str).unique())
            print(f"NB01 pathology patients: {len(nb01_patients)}")
        except Exception as e:
            print(f"NB01 patch_manifest read failed: {e}")
    else:
        print("No patch_manifest.csv found under NB01 root.")

# ---- Cross-check 2: NB02 genomic summary -------------------
nb02_patients = set()
if PATHS["nb02"].exists():
    gen_sum = list(PATHS["nb02"].rglob("genomic_patient_summary.csv"))
    gen_ev  = list(PATHS["nb02"].rglob("genomic_evidence.json"))
    if gen_sum:
        try:
            gs = pd.read_csv(gen_sum[0], usecols=["patient_id"])
            nb02_patients = set(gs["patient_id"].astype(str).unique())
            print(f"NB02 genomic patients:   {len(nb02_patients)}")
        except Exception as e:
            print(f"NB02 genomic_patient_summary read failed: {e}")
    elif gen_ev:
        try:
            with open(gen_ev[0], "r", encoding="utf-8") as f:
                ge = json.load(f)
            nb02_patients = set(ge.keys())
            print(f"NB02 genomic patients (from JSON): {len(nb02_patients)}")
        except Exception as e:
            print(f"NB02 genomic_evidence.json read failed: {e}")
    else:
        print("No NB02 genomic summary found.")

# ---- Cross-check 3: NB00 modality completeness -------------
mod_comp = None
mc_path = PATHS["nb00"] / "modality_completeness.csv"
if mc_path.exists():
    mod_comp = pd.read_csv(mc_path)
    print(f"modality_completeness: {mod_comp.shape}")
    if "patient_id" in mod_comp.columns:
        mod_patients = set(mod_comp["patient_id"].astype(str).unique())
        print(f"  modality_completeness patients: {len(mod_patients)}")
        print(f"  intersection with cohort:       {len(mod_patients & patient_set)}")
else:
    print(f"modality_completeness.csv not found at {mc_path}")

# ---- Save overlap summary ----------------------------------
overlap_rows = []
for pid in sorted(patient_set):
    overlap_rows.append({
        "patient_id":    pid,
        "in_cohort":     True,
        "in_pathology":  pid in nb01_patients if nb01_patients else None,
        "in_genomic":    pid in nb02_patients if nb02_patients else None,
    })
overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv(NB_OUTPUT / "patient_cross_checks.csv", index=False)

print(f"\n[✓] cohort loaded in {time.time()-t0:.1f}s")

cohort_joined_wsi.csv: (478, 9)
cohort: 478 cases | 478 unique patients
NB02 genomic patients:   478
modality_completeness: (478, 6)
  modality_completeness patients: 478
  intersection with cohort:       478

[✓] cohort loaded in 0.1s


## PROCESSING — resolve TCGA clinical files

In [4]:
# ============================================================
# PROCESSING — resolve TCGA clinical files + robust reader
# ============================================================
def resolve_tcga_file(file_name):
    for p in PATHS["tcga"].rglob(file_name):
        if p.is_file():
            return p
    raise FileNotFoundError(f"TCGA file not found: {file_name}")

PATIENT_FILE = resolve_tcga_file("data_clinical_patient.txt")
SAMPLE_FILE  = resolve_tcga_file("data_clinical_sample.txt")
print(f"patient file: {PATIENT_FILE}")
print(f"sample  file: {SAMPLE_FILE}")

def read_cbioportal_clinical(path):
    """Read a cBioPortal clinical file, skipping '#' metadata lines."""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()
    header_idx = None
    for i, line in enumerate(lines):
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        header_idx = i
        break
    if header_idx is None:
        raise ValueError(f"No header line found in {path}")
    df = pd.read_csv(path, sep="\t", header=header_idx,
                     low_memory=False, comment=None, encoding="utf-8")
    df = df[~df.iloc[:, 0].astype(str).str.startswith("#")]
    return df

clinical_patient_df = read_cbioportal_clinical(PATIENT_FILE)
clinical_sample_df  = read_cbioportal_clinical(SAMPLE_FILE)

print(f"\nclinical_patient_df: {clinical_patient_df.shape}")
print(f"clinical_sample_df:  {clinical_sample_df.shape}")
print(f"patient columns ({len(clinical_patient_df.columns)}): {list(clinical_patient_df.columns)}")
print(f"sample  columns ({len(clinical_sample_df.columns)}):  {list(clinical_sample_df.columns)}")

patient file: data\tcga\luad_tcga_pan_can_atlas_2018\data_clinical_patient.txt
sample  file: data\tcga\luad_tcga_pan_can_atlas_2018\data_clinical_sample.txt

clinical_patient_df: (566, 38)
clinical_sample_df:  (566, 19)
patient columns (38): ['PATIENT_ID', 'SUBTYPE', 'CANCER_TYPE_ACRONYM', 'OTHER_PATIENT_ID', 'AGE', 'SEX', 'AJCC_PATHOLOGIC_TUMOR_STAGE', 'AJCC_STAGING_EDITION', 'DAYS_LAST_FOLLOWUP', 'DAYS_TO_BIRTH', 'DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS', 'ETHNICITY', 'FORM_COMPLETION_DATE', 'HISTORY_NEOADJUVANT_TRTYN', 'ICD_10', 'ICD_O_3_HISTOLOGY', 'ICD_O_3_SITE', 'INFORMED_CONSENT_VERIFIED', 'NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT', 'PATH_M_STAGE', 'PATH_N_STAGE', 'PATH_T_STAGE', 'PERSON_NEOPLASM_CANCER_STATUS', 'PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT', 'PRIOR_DX', 'RACE', 'RADIATION_THERAPY', 'WEIGHT', 'IN_PANCANPATHWAYS_FREEZE', 'OS_STATUS', 'OS_MONTHS', 'DSS_STATUS', 'DSS_MONTHS', 'DFS_STATUS', 'DFS_MONTHS', 'PFS_STATUS', 'PFS_MONTHS', 'GENETIC_ANCESTRY_LABEL']
sample  column

## PROCESSING — restrict to cohort and report coverage

In [5]:
# ============================================================
# PROCESSING — restrict to cohort
# ============================================================
if "PATIENT_ID" not in clinical_patient_df.columns:
    raise ValueError("PATIENT_ID missing from clinical_patient_df")

patient_clinical = clinical_patient_df[
    clinical_patient_df["PATIENT_ID"].astype(str).isin(patient_set)
].copy()

sample_clinical = pd.DataFrame()
if "PATIENT_ID" in clinical_sample_df.columns:
    sample_clinical = clinical_sample_df[
        clinical_sample_df["PATIENT_ID"].astype(str).isin(patient_set)
    ].copy()

print(f"cohort clinical patients: {len(patient_clinical)} (expected {len(patient_set)})")
print(f"cohort clinical samples:  {len(sample_clinical)}")

if len(sample_clinical):
    spp = sample_clinical.groupby("PATIENT_ID").size().to_dict()
    print(f"samples per patient (min={min(spp.values())}, max={max(spp.values())}, "
          f"median={int(pd.Series(list(spp.values())).median())})")
    n_patients_with_samples = len(spp)
    print(f"patients with >=1 sample row: {n_patients_with_samples}")
    if max(spp.values()) == 1:
        print("NOTE: 1 representative sample per patient — sample file is "
              "representative-level, not one row per WSI.")

missing_clin = sorted(patient_set - set(patient_clinical["PATIENT_ID"].astype(str)))
if missing_clin:
    print(f"WARNING: no clinical record for {len(missing_clin)} patients:")
    print(f"  {missing_clin[:20]}{'...' if len(missing_clin)>20 else ''}")
else:
    print(f"all {len(patient_set)} patients have clinical records")

cohort clinical patients: 471 (expected 478)
cohort clinical samples:  471
samples per patient (min=1, max=1, median=1)
patients with >=1 sample row: 471
NOTE: 1 representative sample per patient — sample file is representative-level, not one row per WSI.
  ['TCGA-05-4245', 'TCGA-05-5420', 'TCGA-38-4629', 'TCGA-38-4630', 'TCGA-44-2661', 'TCGA-44-2664', 'TCGA-44-6147']


## PROCESSING — per-patient structured clinical extraction

In [6]:
# ============================================================
# PROCESSING — cleaning helpers and field targets
# ============================================================
EMPTY_VALUES = {"", "na", "n/a", "nan", "[not available]", "[unknown]",
                "[not applicable]", "[not evaluated]", "not available",
                "unknown", "none", "."}

def get_clean(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s.lower() in EMPTY_VALUES:
        return None
    return s

patient_col_map = {str(c).upper(): c for c in patient_clinical.columns}
sample_col_map  = {str(c).upper(): c for c in sample_clinical.columns} if len(sample_clinical) else {}

def resolve_column(field, col_map):
    return col_map.get(field.upper())

# Patient-file fields — expanded to match the broader TCGA schema
PATIENT_FIELDS = {
    "demographics": [
        "AGE", "SEX", "RACE", "ETHNICITY", "WEIGHT",
        "GENETIC_ANCESTRY_LABEL",
    ],
    "staging": [
        "AJCC_PATHOLOGIC_TUMOR_STAGE", "AJCC_STAGING_EDITION",
        "PATH_T_STAGE", "PATH_N_STAGE", "PATH_M_STAGE", "TUMOR_STAGE",
    ],
    "diagnosis": [
        "CANCER_TYPE_ACRONYM", "ICD_10", "ICD_O_3_HISTOLOGY",
        "ICD_O_3_SITE", "SUBTYPE", "PRIOR_DX",
    ],
    "survival": [
        "OS_STATUS", "OS_MONTHS",
        "DSS_STATUS", "DSS_MONTHS",
        "DFS_STATUS", "DFS_MONTHS",
        "PFS_STATUS", "PFS_MONTHS",
    ],
    "followup": [
        "DAYS_LAST_FOLLOWUP",
        "DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS",
        "DAYS_TO_BIRTH",
    ],
    "treatment": [
        "HISTORY_NEOADJUVANT_TRTYN",
        "PRIOR_TREATMENT", "TREATMENT_TYPE", "INITIAL_TREATMENT",
        "RADIATION_THERAPY",
    ],
    "events": [
        "NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT",
        "PERSON_NEOPLASM_CANCER_STATUS",
    ],
}

SAMPLE_FIELDS = {
    "sample_diagnosis": [
        "CANCER_TYPE", "CANCER_TYPE_DETAILED", "ONCOTREE_CODE",
        "TUMOR_TYPE", "HISTOLOGY", "GRADE",
    ],
    "sample_biomarkers": [
        "MSI_SCORE_MANTIS", "MSI_SCORE_MANTISS",
        "MSI_SENSOR_SCORE",
        "ANEUPLOIDY_SCORE",
        "TMB_NONSYNONYMOUS", "TMB",
        "TBL_SCORE",
    ],
    "sample_meta": [
        "SAMPLE_TYPE", "TUMOR_TISSUE_SITE",
        "TISSUE_SOURCE_SITE", "TISSUE_SOURCE_SITE_CODE",
        "SOMATIC_STATUS",
        "TISSUE_PROSPECTIVE_COLLECTION_INDICATOR",
        "TISSUE_RETROSPECTIVE_COLLECTION_INDICATOR",
    ],
}

ALL_SECTIONS = ["demographics", "staging", "diagnosis", "survival",
                "followup", "treatment", "events",
                "sample_diagnosis", "sample_biomarkers", "sample_meta"]

In [7]:
# ============================================================
# PROCESSING — per-patient structured clinical extraction
# ============================================================
clinical_structured = {}
extraction_qc = []

for pid in sorted(patient_set):
    row_df = patient_clinical[patient_clinical["PATIENT_ID"].astype(str) == pid]
    if len(row_df) == 0:
        clinical_structured[pid] = {"patient_id": pid, "record_found": False}
        extraction_qc.append({"patient_id": pid, "record_found": False,
                              "fields_extracted": 0, "n_samples": 0})
        continue

    prow = row_df.iloc[0]
    record = {"patient_id": pid, "record_found": True}

    # Patient-file fields
    for section, fields in PATIENT_FIELDS.items():
        record[section] = {}
        for f in fields:
            col = resolve_column(f, patient_col_map)
            if col is not None and col in prow.index:
                v = get_clean(prow[col])
                if v is not None:
                    record[section][f] = v

    # Initialize sample-derived sections
    record["samples"] = []
    record["sample_diagnosis"] = {}
    record["sample_biomarkers"] = {}
    record["sample_meta"] = {}

    # Sample-file fields
    if len(sample_clinical):
        srow_df = sample_clinical[sample_clinical["PATIENT_ID"].astype(str) == pid]
        if len(srow_df):
            record["samples"] = srow_df.to_dict(orient="records")
            srow = srow_df.iloc[0]   # representative sample
            for section, fields in SAMPLE_FIELDS.items():
                for f in fields:
                    col = resolve_column(f, sample_col_map)
                    if col is not None and col in srow.index:
                        v = get_clean(srow[col])
                        if v is not None:
                            record[section][f] = v

    # Preserve raw patient row for provenance
    record["raw_fields"] = {
        str(k): (None if pd.isna(v) else str(v)) for k, v in prow.items()
    }

    clinical_structured[pid] = record

    n_extracted = sum(
        1 for section in ALL_SECTIONS
        for v in record.get(section, {}).values()
        if v is not None
    )
    extraction_qc.append({
        "patient_id": pid,
        "record_found": True,
        "fields_extracted": n_extracted,
        "n_samples": len(record["samples"]),
    })

print(f"\nstructured clinical records: {len(clinical_structured)}")
for pid, rec in list(clinical_structured.items())[:10]:
    if not rec.get("record_found"):
        print(f"  {pid}: no clinical record")
        continue
    counts = {s: sum(1 for v in rec.get(s, {}).values() if v is not None)
              for s in ALL_SECTIONS}
    total = sum(counts.values())
    print(f"  {pid}: {total:>3} fields | {counts}")
if len(clinical_structured) > 10:
    print(f"  ... and {len(clinical_structured) - 10} more")


structured clinical records: 478
  TCGA-05-4244:  41 fields | {'demographics': 3, 'staging': 5, 'diagnosis': 6, 'survival': 6, 'followup': 3, 'treatment': 1, 'events': 1, 'sample_diagnosis': 4, 'sample_biomarkers': 5, 'sample_meta': 7}
  TCGA-05-4245: no clinical record
  TCGA-05-4249:  43 fields | {'demographics': 3, 'staging': 5, 'diagnosis': 6, 'survival': 6, 'followup': 3, 'treatment': 2, 'events': 2, 'sample_diagnosis': 4, 'sample_biomarkers': 5, 'sample_meta': 7}
  TCGA-05-4250:  38 fields | {'demographics': 3, 'staging': 5, 'diagnosis': 6, 'survival': 5, 'followup': 2, 'treatment': 1, 'events': 0, 'sample_diagnosis': 4, 'sample_biomarkers': 5, 'sample_meta': 7}
  TCGA-05-4382:  45 fields | {'demographics': 3, 'staging': 5, 'diagnosis': 6, 'survival': 8, 'followup': 3, 'treatment': 2, 'events': 2, 'sample_diagnosis': 4, 'sample_biomarkers': 5, 'sample_meta': 7}
  TCGA-05-4384:  42 fields | {'demographics': 3, 'staging': 5, 'diagnosis': 6, 'survival': 6, 'followup': 3, 'treatment

## PROCESSING — flat clinical table and missingness

In [8]:
import gc
# ============================================================
# PROCESSING — flat clinical table + missingness
# ============================================================
flat_rows = []
for pid, rec in clinical_structured.items():
    row = {"patient_id": pid}
    for section in ALL_SECTIONS:
        for k, v in rec.get(section, {}).items():
            row[f"{section}__{k}"] = v
    row["n_samples"] = len(rec.get("samples", []))
    row["record_found"] = bool(rec.get("record_found", False))
    flat_rows.append(row)

clinical_table = pd.DataFrame(flat_rows)

missing_rows = []
for col in clinical_table.columns:
    if col in ("patient_id",):
        continue
    n_total = len(clinical_table)
    n_missing = int(clinical_table[col].isna().sum())
    missing_rows.append({
        "field": col,
        "n_total": n_total,
        "n_missing": n_missing,
        "pct_missing": round(100.0 * n_missing / n_total, 1) if n_total else 0.0,
    })
missingness_df = pd.DataFrame(missing_rows).sort_values("pct_missing", ascending=False)

high_missing = missingness_df[missingness_df["pct_missing"] > HIGH_MISSINGNESS_THRESHOLD]
print(f"clinical_table: {clinical_table.shape}")
print(f"fields with >{HIGH_MISSINGNESS_THRESHOLD}% missingness: {len(high_missing)}")
if len(high_missing):
    print(high_missing.to_string(index=False))

gc.collect()
print(f"\n[✓] extraction complete")

clinical_table: (478, 50)
fields with >50.0% missingness: 0

[✓] extraction complete


## PROCESSING — minimal clinical feature schema for downstream planners

The downstream planners (Notebook 06) and safety gate (Notebook 08) need a
small, stable set of clinical features. We derive those here.

Every feature is optional. If a source field is missing, the feature stays
null. The `clinical_features.csv` file records which features were populated
per patient.

In [9]:
# ============================================================
# PROCESSING — minimal clinical feature schema
# ============================================================
STAGE_RANK = {
    "stage ia": 1, "stage ib": 1,
    "stage iia": 2, "stage iib": 2,
    "stage iiia": 3, "stage iiib": 3,
    "stage iiic": 3,
    "stage iva": 4, "stage ivb": 4,
    "stage i": 1, "stage ii": 2, "stage iii": 3, "stage iv": 4,
}

def rank_stage(s):
    if s is None:
        return None
    return STAGE_RANK.get(str(s).strip().lower())

def to_float(v):
    if v is None:
        return None
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def os_status_to_int(v):
    if v is None:
        return None
    s = str(v).upper()
    if "DECEASED" in s:
        return 1
    if "LIVING" in s:
        return 0
    return None

feature_rows = []
for pid, rec in clinical_structured.items():
    if not rec.get("record_found"):
        feature_rows.append({"patient_id": pid, "record_found": False})
        continue

    demo = rec.get("demographics", {})
    stag = rec.get("staging", {})
    dx   = rec.get("diagnosis", {})
    sdx  = rec.get("sample_diagnosis", {})
    sbm  = rec.get("sample_biomarkers", {})
    meta = rec.get("sample_meta", {})
    surv = rec.get("survival", {})
    fup  = rec.get("followup", {})
    trt  = rec.get("treatment", {})

    stage_str = stag.get("AJCC_PATHOLOGIC_TUMOR_STAGE") or stag.get("TUMOR_STAGE")

    feature_rows.append({
        "patient_id":       pid,
        "record_found":     True,
        "age":              to_float(demo.get("AGE")),
        "sex":              demo.get("SEX"),
        "race":             demo.get("RACE"),
        "ethnicity":        demo.get("ETHNICITY"),
        "stage_str":        stage_str,
        "stage_rank":       rank_stage(stage_str),
        "path_t_stage":     stag.get("PATH_T_STAGE"),
        "path_n_stage":     stag.get("PATH_N_STAGE"),
        "path_m_stage":     stag.get("PATH_M_STAGE"),
        "cancer_type_acronym": dx.get("CANCER_TYPE_ACRONYM"),
        "icd_10":           dx.get("ICD_10"),
        "subtype":          dx.get("SUBTYPE") or sdx.get("CANCER_TYPE_DETAILED"),
        "oncotree_code":    sdx.get("ONCOTREE_CODE"),
        "tumor_type":       sdx.get("TUMOR_TYPE"),
        "histology":        sdx.get("HISTOLOGY"),
        "grade":            sdx.get("GRADE"),
        "sample_type":      meta.get("SAMPLE_TYPE"),
        "tumor_tissue_site": meta.get("TUMOR_TISSUE_SITE"),
        "msi_score":        to_float(sbm.get("MSI_SCORE_MANTIS") or sbm.get("MSI_SCORE_MANTISS") or sbm.get("MSI_SENSOR_SCORE")),
        "aneuploidy_score": to_float(sbm.get("ANEUPLOIDY_SCORE")),
        "tmb":              to_float(sbm.get("TMB_NONSYNONYMOUS") or sbm.get("TMB")),
        "tbl_score":        to_float(sbm.get("TBL_SCORE")),
        "os_status":        os_status_to_int(surv.get("OS_STATUS")),
        "os_months":        to_float(surv.get("OS_MONTHS")),
        "dss_status":       os_status_to_int(surv.get("DSS_STATUS")),
        "dss_months":       to_float(surv.get("DSS_MONTHS")),
        "pfs_status":       os_status_to_int(surv.get("PFS_STATUS")),
        "pfs_months":       to_float(surv.get("PFS_MONTHS")),
        "days_last_followup": to_float(fup.get("DAYS_LAST_FOLLOWUP")),
        "neo_adjuvant_treatment": trt.get("HISTORY_NEOADJUVANT_TRTYN"),
        "radiation_therapy":      trt.get("RADIATION_THERAPY"),
    })

clinical_features = pd.DataFrame(feature_rows)

# Report populated fraction per feature
feat_pop = (
    clinical_features
    .drop(columns=["patient_id"])
    .notna()
    .mean()
    .sort_values(ascending=False)
    .rename("populated_fraction")
    .reset_index()
    .rename(columns={"index": "feature"})
)
display(feat_pop)

clinical_features.to_csv(NB_OUTPUT / "clinical_features.csv", index=False)
print(f"Saved: clinical_features.csv ({clinical_features.shape})")

,feature,populated_fraction
0,record_found,1.000000
1,sex,0.985356
2,path_t_stage,0.985356
3,tumor_type,0.985356
4,subtype,0.985356
5,oncotree_code,0.985356
6,cancer_type_acronym,0.985356
7,icd_10,0.985356
8,tmb,0.985356
9,os_status,0.985356


Saved: clinical_features.csv ((478, 33))


## EVALUATION — QC report + dashboard

In [10]:
# ============================================================
# EVALUATION — QC report + dashboard
# ============================================================
qc = {
    "notebook":               NOTEBOOK,
    "timestamp_utc":          datetime.now(timezone.utc).isoformat(),
    "patients_in_cohort":     int(len(patient_set)),
    "patients_with_clinical": int(sum(1 for r in clinical_structured.values()
                                      if r.get("record_found", False))),
    "total_fields_available": int(len(clinical_table.columns) - 2),  # minus id, record_found
    "fields_fully_populated": int((missingness_df["pct_missing"] == 0).sum()),
    "fields_mostly_missing":  int((missingness_df["pct_missing"] > HIGH_MISSINGNESS_THRESHOLD).sum()),
    "high_missingness_threshold": float(HIGH_MISSINGNESS_THRESHOLD),
    "sections_extracted":     ALL_SECTIONS,
    "n_samples_total":        int(sum(r.get("n_samples", 0) for r in clinical_structured.values())),
    "patients_in_nb01":       int(len(nb01_patients)) if nb01_patients else None,
    "patients_in_nb02":       int(len(nb02_patients)) if nb02_patients else None,
    "patient_cross_check_cohort_nb01": int(len(patient_set & nb01_patients)) if nb01_patients else None,
    "patient_cross_check_cohort_nb02": int(len(patient_set & nb02_patients)) if nb02_patients else None,
    "patient_leakage_note":   "Clinical evidence joins on patient_id; no independent splits",
    "missingness_by_field":   missingness_df.to_dict(orient="records"),
    "feature_population":     feat_pop.to_dict(orient="records"),
}
with open(NB_OUTPUT / "clinical_qc_report.json", "w", encoding="utf-8") as f:
    json.dump(qc, f, indent=2, default=str)

# ---- Dashboard ----
try:
    from IPython.display import Image, display as _display
except ImportError:
    Image = lambda *a, **kw: None
    _display = lambda *a, **kw: None

fig, axes = plt.subplots(2, 2, figsize=(18, 11))

# 1. Top missingness
top_miss = missingness_df.head(20)
axes[0, 0].barh(range(len(top_miss)), top_miss["pct_missing"].values, color="steelblue")
axes[0, 0].set_yticks(range(len(top_miss)))
axes[0, 0].set_yticklabels(top_miss["field"].str.replace("__", " | ", regex=False),
                           fontsize=6)
axes[0, 0].invert_yaxis()
axes[0, 0].set_xlabel("% missing")
axes[0, 0].set_title("Top 20 fields by missingness")
axes[0, 0].axvline(HIGH_MISSINGNESS_THRESHOLD, color="red",
                   linestyle="--", alpha=0.5, label=f"{HIGH_MISSINGNESS_THRESHOLD}%")
axes[0, 0].legend()

# 2. Fields populated per patient
qc_df = pd.DataFrame(extraction_qc)
if len(qc_df):
    qc_df = qc_df.sort_values("fields_extracted", ascending=False).reset_index(drop=True)
    colors = ["darkorange" if f >= 20 else "crimson" for f in qc_df["fields_extracted"]]
    axes[0, 1].bar(range(len(qc_df)), qc_df["fields_extracted"].values, color=colors, width=0.9)
    if len(qc_df) > 30:
        step = 50 if len(qc_df) <= 500 else 100
        ticks = list(range(0, len(qc_df), step))
        if (len(qc_df) - 1) not in ticks:
            ticks.append(len(qc_df) - 1)
        axes[0, 1].set_xticks(ticks)
        axes[0, 1].set_xticklabels([str(t + 1) for t in ticks], fontsize=8)
        axes[0, 1].set_xlabel(f"Patient Rank (1 to {len(qc_df)}, sorted by completeness)")
    else:
        axes[0, 1].set_xticks(range(len(qc_df)))
        axes[0, 1].set_xticklabels(qc_df["patient_id"], rotation=45, ha="right", fontsize=7)
        axes[0, 1].set_xlabel("Patient ID")
    med_fields = float(qc_df["fields_extracted"].median())
    axes[0, 1].axhline(med_fields, color="navy", linestyle="--", linewidth=1.2, label=f"Median ({int(med_fields)} fields)")
    n_missing_records = int((qc_df["fields_extracted"] == 0).sum())
    if n_missing_records > 0:
        axes[0, 1].plot([], [], color="crimson", label=f"No clinical record ({n_missing_records} patients)")
    axes[0, 1].legend(loc="lower left", fontsize=8, framealpha=0.9)
    axes[0, 1].set_ylabel("Fields populated")
    axes[0, 1].set_title(f"Clinical fields extracted per patient (N={len(qc_df)}, Median={int(med_fields)})")

# 3. Age distribution (if available)
if "age" in clinical_features.columns and clinical_features["age"].notna().any():
    axes[1, 0].hist(clinical_features["age"].dropna(), bins=20, color="seagreen", alpha=0.8)
    axes[1, 0].set_xlabel("Age (years)")
    axes[1, 0].set_title("Age distribution")
else:
    axes[1, 0].text(0.5, 0.5, "No age data", ha="center", va="center")
    axes[1, 0].set_title("Age distribution")

# 4. Stage distribution
if "stage_rank" in clinical_features.columns and clinical_features["stage_rank"].notna().any():
    sr = clinical_features["stage_rank"].value_counts().sort_index()
    axes[1, 1].bar(sr.index.astype(int), sr.values, color="mediumpurple")
    axes[1, 1].set_xticks(sr.index.astype(int))
    axes[1, 1].set_xticklabels([f"Stage {int(i)}" for i in sr.index])
    axes[1, 1].set_ylabel("# patients")
    axes[1, 1].set_title("AJCC pathologic stage distribution")
else:
    axes[1, 1].text(0.5, 0.5, "No stage data", ha="center", va="center")
    axes[1, 1].set_title("Stage distribution")

plt.tight_layout()
dash_path = NB_OUTPUT / "clinical_qc_dashboard.png"
plt.savefig(dash_path, dpi=180, bbox_inches="tight")
_display(Image(filename=str(dash_path)))
plt.close(fig)
print("Saved:", dash_path)

# Summary table
display(pd.DataFrame([
    {"metric": k, "value": (str(v)[:160] if isinstance(v, (dict, list)) else v)}
    for k, v in qc.items()
]))

,metric,value
0,notebook,03_clinical_extraction
1,timestamp_utc,2026-10-19T13:20:38.727625+00:00
2,patients_in_cohort,478
3,patients_with_clinical,471
4,total_fields_available,48
5,fields_fully_populated,2
6,fields_mostly_missing,0
7,high_missingness_threshold,50.0
8,sections_extracted,"['demographics', 'staging', 'diagnosis', 'survival', 'followup', 'treatment', 'events', 'sample_diagnosis', 'sample_biomarkers', 'sample_meta']"
9,n_samples_total,0


Saved: VERIONCO_outputs\03\clinical_qc_dashboard.png


In [11]:
# ============================================================
# EXPORT & DOWNLOAD ARTIFACT BUNDLE
# ============================================================
import shutil
from pathlib import Path
try:
    from IPython.display import FileLink, display
except ImportError:
    def FileLink(p): return p
    def display(*a): pass

export_dir = OUTPUT_ROOT / 'exports'
export_dir.mkdir(parents=True, exist_ok=True)
zip_name = f'verionco_outputs_{NB_OUTPUT.name}'
zip_path = export_dir / zip_name

# Package NB_OUTPUT into zip
shutil.make_archive(str(zip_path), 'zip', NB_OUTPUT)
final_zip = export_dir / f'{zip_name}.zip'

print('=' * 60)
print(f'VERIONCO NOTEBOOK {NB_OUTPUT.name} COMPLETE')
print(f'Artifact bundle created: {final_zip}')
print(f'Archive size: {final_zip.stat().st_size / (1024*1024):.2f} MB')
print('=' * 60)

# Display download link (clickable in Kaggle/Jupyter)
display(FileLink(str(final_zip)))


VERIONCO NOTEBOOK 03 COMPLETE
Artifact bundle created: VERIONCO_outputs\exports\verionco_outputs_03.zip
Archive size: 0.28 MB
